# Séance 8 - Le Bivarié I

Voir user guide :
- Marks (2/3) : https://altair-viz.github.io/user_guide/marks/index.html

## Objectifs
- Visualiser une relation entre deux variables, ou plus
- Comprendre l'intérêt des nuages de points
- Comprendre la différence entre l'utilisation de diagrammes en barres ou lignes
- Créer des **groupes** avec **pd.cut**

Rappel : Nous utilisons **pandas** pour préparer, transformer et résumer les données.

## Pourquoi le bivarié ?

L'analyse bivariée permet d'explorer la **relation entre deux variables**. Cela nous permet de répondre à des questions plus intéressantes comme :
- Le niveau d'éducation influence-t-il le revenu ?
- L'âge est-il lié à la préférence partisane ?
- La polarisation politique varie-t-elle selon les groupes sociaux ?

**Bonnes pratiques** :
- Toujours se demander : quelle variable expliquer (Y) par quelle variable explicative (X) ?
- Choisir le type de graphique selon la nature des variables (Q/Q, Q/O, O/O)
- Ajouter des éléments de contexte (titre, exemple de lecture, source)

In [ ]:
# ===========================================
# Installation & Chargement des bibliothèques
# ===========================================
%pip install "vegafusion[embed]>=1.5.0" "vl-convert-python>=1.6.0"

import pandas as pd
import altair as alt
import warnings

warnings.filterwarnings("ignore")

# Configuration d'Altair
alt.data_transformers.enable("vegafusion")  # exécution locale
alt.data_transformers.disable_max_rows()  # dataset large

In [ ]:
# ===========================================
# Chargement de la base de données
# ===========================================
data_url = "https://raw.githubusercontent.com/datamisc/ts-2024/main/data.csv"
df_raw = pd.read_csv(data_url, compression="gzip", low_memory=False)

In [ ]:
# Sélectionner quelques variables d'intérêt
my_vars = [
    "V241156",  # thermomètre Harris
    "V241157",  # thermomètre Trump
    "V241177",  # idéologie
    "V241043",  # intention de vote
    "V241465x",  # éducation
    "V241566x",  # revenu (19 catégories - recodage nécessaire)
    "V241567x",  # revenu (6 catégories - déjà codé)
    "V241458x",  # âge
    "V241501x",  # ethnie
    "V241550",  # sex,
]

df = df_raw[my_vars].copy()

df.columns = [
    "thermo_harris",
    "thermo_trump",
    "ideology",
    "vote_int",
    "education",
    "income_raw",
    "income_cat",
    "age",
    "ethnicity",
    "sex",
]

df.head()

## Nuage de points (scatter plot)

### Quand l'utiliser ?
- Pour explorer la relation entre **deux variables quantitatives/continues**
- Permet de visualiser la forme de la relation (linéaire, exponentielle, etc.)
- Utile pour détecter des valeurs aberrantes ou des clusters
### Bonnes pratiques :
- Utiliser la transparence (`opacity=...`) pour gérer la superposition des points
- Utiliser des couleurs/formes (`shape=...` et/ou `color=...`) pour représenter d'autres dimensions
- Nous pouvons aussi ajouter une droite de régression linéaire pour montrer la tendance (plus tard...)

**Exemple** : Relation entre les thermomètres de sympathie Harris et Trump

In [ ]:
# Préparation des données
mask = (df["thermo_harris"].between(0, 100)) & (df["thermo_trump"].between(0, 100))
df_scatter = df[mask]
vars = ["thermo_harris", "thermo_trump", "vote_int"]
df_scatter = df[vars]

df_scatter

In [ ]:
alt.Chart(df_scatter).mark_point(size=25, opacity=0.35).encode(
    x=alt.X(
        "thermo_trump",
        type="quantitative",
        title="Thermomètre Trump",
        scale=alt.Scale(domain=[0, 100]),
    ),
    y=alt.Y(
        "thermo_harris",
        type="quantitative",
        title="Thermomètre Harris",
        scale=alt.Scale(domain=[0, 100]),
    ),
).properties(
    title=alt.TitleParams(
        text="Évaluations de Harris et Trump sur une échelle de sympathie",
        subtitle=[
            # Mini hack-time: comment faire en sorte que le sous-titre aille à la ligne?
            "Les points en haut à gauche représentent les personnes qui ont une opinion très favorable de Harris et très défavorable de Trump.",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=400,
    height=400,
)

**Interprétation** : On observe une relation négative, plus les répondants ont tendance à apprécier Trump et moins ils semblent apprécier Harris. 

### Hack-Time 

Ajoutez une couleur selon l'intention de vote (`vote_int`):
- Conservez uniquement les valeurs entre 1 et 3 pour `vote_int`

Puis utilisez le paramètre:
- `color = alt.Color('vote_int', type='nominal')`

In [ ]:
# Hack-Time : votre code ici


## Graphique en barres (bar plot)

### Quand l'utiliser ?
- Pour visualiser une relation entre une **variable catégorielle (X)** et une **variable quantitative (Y)**
- Préférable quand les catégories ne sont pas ordonnées 
- Plus lisible pour un petit nombre de catégories

**Exemple** : Revenu moyen selon l'origine ethnique
- La variable `income_raw` (V241566x) contient 28 catégories de revenu. 
- La variable `ethnicity` (V241501x) contient 6 catégories (concentrons nous sur les 3 premières).

In [ ]:
# Verifier les valeurs de income_raw
df["income_raw"].value_counts().sort_index()

Maintenant, calculons le revenu moyen par origine ethnique et visualisons avec un barplot :

In [ ]:
# Calcul des moyennes de revenu par éducation

df_inc = df[(df["ethnicity"].between(1, 3)) & (df["income_raw"].between(1, 28))][
    ["ethnicity", "income_raw"]
]

df_means = df_inc.groupby("ethnicity", as_index=False)["income_raw"].mean()

df_means["ethnicity"] = df_means["ethnicity"].replace(
    {1: "White", 2: "Black", 3: "Hispanic"}
)

df_means

In [ ]:
alt.Chart(df_means).mark_bar(color="#BE8400").encode(
    x=alt.X("ethnicity", type="nominal", title=""),
    y=alt.Y("income_raw", type="quantitative", title="Code de Revenu Moyen"),
).properties(
    title=alt.TitleParams(
        text="Revenu moyen du foyer par origine ethnique",
        subtitle=[
            "Le revenu moyen des hispaniques se situe entre 55 000 $ et 59 999 $.",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=300,
    height=300,
)

## Graphiques linéaires (line plot)

### Quand l'utiliser ?
- Également pour une relation catégorielle -> quantitative
- Et lorsque nous souhaitons montrer des **tendances** ou des **évolutions**
- Permet de comparer facilement plusieurs groupes
### Bonnes pratiques :
- Bien ordonner les catégories sur l'axe X (ne pas se fier à l'ordre par défaut)
- Ajuster les points pour montrer les valeurs exactes
- Utiliser des couleurs/formes pertinentes

**Exemple** : Évaluation moyenne de Harris selon l'idéologie politique

In [ ]:
# Calcul des moyennes avec pandas
mask = (
    (df["ideology"].between(1, 7))
    & df["thermo_harris"].between(0, 100)
    & (df["thermo_trump"].between(0, 100))
)
vars = ["ideology", "thermo_harris", "thermo_trump"]
df_mean = df[mask][vars]

means = df_mean.groupby("ideology", as_index=False).mean()

means

In [ ]:
alt.Chart(means).mark_line(point=True, size=3, color="#1D4ED8").encode(
    x=alt.X("ideology", type="ordinal", title="Idéologie (1=libéral, 7=conservateur)"),
    y=alt.Y(
        "thermo_harris",
        type="quantitative",
        title="Thermomètre de sympathie",
        scale=alt.Scale(domain=[0, 100]),
    ),
).properties(
    title=alt.TitleParams(
        text="Évolution moyenne de la sympathie envers Harris selon l'idéologie",
        subtitle=[
            "Les répondants d'idéologie 1-2 donnent en moyenne ~80 au thermomètre Harris.",
            "Source : ANES 2024 Time Series Study",
        ],
        anchor="start",
    ),
    width=520,
    height=300,
)

**Interprétation** : Plus on va vers la droite politique, plus l'évaluation moyenne de Harris diminue.

### Hack-Time

Reproduisez le même graphique mais pour Trump (`thermo_trump`).
- calculez la moyenne du thermomètre Trump par idéologie avec pandas
- tracez la ligne de moyennes 
- écrivez un titre + sous-titre avec exemple de lecture ou interprétation

Bonus : Tracez les deux lignes (Harris et Trump) sur le même graphique pour comparer :
- Créez un dataframe avec les deux moyennes
- Utilisez `color` pour distinguer Harris et Trump

In [ ]:
# Hack-Time : votre code ici


Pour un public scientifique/académique vous pouvez également utiliser une boîte à moustaches.

In [ ]:
df_box = df[["ideology", "thermo_harris"]]
df_box = df_box[df_box["ideology"].between(1, 7)]
df_box = df_box[df_box["thermo_harris"] > 0]

alt.Chart(df_box).mark_boxplot().encode(
    x=alt.X("ideology", type="ordinal", title="Idéologie (1=libéral, 7=conservateur)"),
    y=alt.Y(
        "thermo_harris",
        type="quantitative",
        title="Thermomètre Harris",
        scale=alt.Scale(domain=[0, 100]),
    ),
).properties(
    title=alt.TitleParams(
        text="Évolution de la sympathie envers Harris selon l'idéologie",
        subtitle=["Source : ANES 2024 Time Series Study"],
        anchor="start",
    ),
    width=520,
    height=300,
)

### Line vs Barplot vs Boxplot ?
- **Line** : préférer pour des catégories ordonnées (idéologie, âge, revenu)
- **Barplot** : préférer pour des catégories sans ordre naturel (région, parti, ethnie)
- **Boxplot** : préférer pour un public plus scientifique

De manière générale
- Si votre objectif principal est de comparer des catégories utilisez un bar plot.
- Si vous voulez mettre l’accent sur la tendance et l’augmentation/diminution graduelle utilisez un line plot.


## Est-ce utile d'avoir un diplôme?


In [ ]:
# Filtrage et nettoyage d'une partie des données
vars = ["education", "income_raw"]
df_educ = df[vars]

df_educ = df[df["education"].between(1, 5)]
df_educ = df_educ[df_educ["income_raw"] > 0]

education_labels = {
    1: "1. Sans diplôme",
    2: "2. Diplôme secondaire",
    3: "3. Études supérieures partielles",
    4: "4. Licence/Bac+3",
    5: "5. Master/Doctorat",
}

df_educ["education"] = df_educ["education"].replace(education_labels)
df_educ["income_raw"].value_counts()


In [ ]:
# Calcul des moyennes
income_means = df_educ.groupby("education", as_index=False)["income_raw"].mean()
income_means

In [ ]:
my_title = alt.TitleParams(
    text="Évolution du revenu en fonction du niveau d'éducation",
    subtitle=[
        "Le revenu moyen augmente avec l'éducation.",
        "Source : ANES 2024 Time Series Study",
    ],
    anchor="start",
)


alt.Chart(income_means).mark_line(
    color="#059669",
    point=True,  # ajoute des points -> plus lisible
).encode(
    x=alt.X(
        "education",
        type="ordinal",
        title="",
        # axis=alt.Axis(labelAngle=30)
    ),
    y=alt.Y("income_raw", type="quantitative", title="Revenus (10=28k$; 24=95k$)"),
).properties(title=my_title, width=300, height=300)

Comme nous l'avons vu, la variable `income_raw` (V241566x) contient 28 catégories de revenu... C'est beaucoup! 
Nous pouvons la recoder en groupes plus lisibles avec **pd.cut** :

In [ ]:
# Création de catégories de revenu avec pd.cut
# Les codes 1-19 correspondent à des tranches de revenus

income_labels = [
    "Moins de 20k",
    "$20k-$-$49.9k",
    "$50k-$99.9k",
    "$100k-$149.9k",
    "$150k-$249.9k",
    "$250k et plus",
]

# On utilise les bins pour les codes 1-15
df["income_recode"] = pd.cut(
    df["income_raw"], bins=[0, 6, 14, 22, 25, 27, 28], labels=income_labels
)

# Verification
df["income_recode"].value_counts()

**Mais**, nous pouvons aussi, en nous référant au notebook de l'ANES, voir qu'il existe déjà des variables regroupées ! 
La variable V241567x est déjà codée en 6 catégories que nous pouvons utiliser.

Il nous suffit alors uniquement d'ajuster les labels et nous allons voir une nouvelle manière de le faire !

In [ ]:
# Calcul des moyennes avec income_cat
df_income_2 = df[(df["education"].between(1, 5)) & (df["income_cat"].between(1, 6))][
    ["education", "income_cat", "sex"]
]

income_means_2 = df_income_2.groupby("education", as_index=False)["income_cat"].mean()

income_means_2["education"] = income_means_2["education"].replace(education_labels)
income_means_2

In [ ]:
# Une version plus avancée...
alt.Chart(income_means_2).mark_line(
    color="#059669",
    point=True,
).encode(
    x=alt.X(
        "education",
        type="ordinal",
        title="",
        # axis=alt.Axis(labelAngle=30)
    ),
    y=alt.Y(
        "income_cat",
        type="quantitative",
        title="",
        scale=alt.Scale(domain=[0, 6]),
        axis=alt.Axis(  # C'est un exemple de recodage d'axe avancé (pour plus tard ...)
            values=[1, 2, 3, 4, 5, 6],
            labelExpr="""
                datum.value == 1 ? 'Moins de $25k' :
                datum.value == 2 ? '$25k-$50k' :
                datum.value == 3 ? '$50k-$100k' :
                datum.value == 4 ? '$100k-$150k' :
                datum.value == 5 ? '$150k-$250k' :
                '$250k et plus'
            """,
        ),
    ),
).properties(title=my_title, width=300, height=300)

### Hack-Time

**Exercice A** : Représentez la moyenne du revenu (`income_cat`) par niveau d'éducation avec un **barplot** au lieu d'un lineplot.

**Exercice B** : Comment le revenu évolue-t-il selon le sexe ?

Ajoutez : titre + sous-titre avec exemple de lecture ou interprétation + source.

In [ ]:
# Hack-Time : votre code ici


## Polarisation et âge : qui sont les plus polarisés?

### Création d'une variable composite

La **polarisation** peut être mesurée par la difference absolue entre les thermomètres de sympathie Trump et Harris. Cela capture l'intensité de la préférence politique (peu importe le candidat préféré, les personnes polarisées ont une opinion très tranchée).

In [ ]:
# Création de la variable polarisation (valeur ABSOLUE de la difference)
df["polarisation"] = abs(df["thermo_trump"] - df["thermo_harris"])

# Verification
df["polarisation"].describe()

### Utilisation de pd.cut pour les groupes d'âge

In [24]:
# Création de catégories d'age avec pd.cut
df["age_group"] = pd.cut(
    df["age"],
    bins=[17, 29, 44, 59, 74, 100],
    labels=["18-29 ans", "30-44 ans", "45-59 ans", "60-74 ans", "75+ ans"],
)

In [ ]:
# Filtre et sélection de quelques variables
mask = df["polarisation"].between(0, 100)
df_pol = df[mask]
vars = ["age_group", "polarisation", "sex"]
df_pol = df_pol[vars]
df_pol

In [ ]:
# Calcul de la polarisation moyenne par groupe d'âge
pol_means = df_pol.groupby("age_group", as_index=False)["polarisation"].mean()
pol_means

In [ ]:
alt.Chart(pol_means).mark_line(color="#7C3AED").encode(
    x=alt.X("age_group", type="nominal", title=""),
    y=alt.Y("polarisation", type="quantitative", title="Polarisation"),
).properties(
    title=alt.TitleParams(
        text="Polarisation politique selon l'âge",
    ),
    width=520,
    height=300,
)

### Hack-Time 

Améliorer le graphique ci-dessus !
Vous pouvez modifier directement le code précédent ou créer une nouvelle cellule de code ci-dessous.